# Lesson 6 — Interpreting and Naming Segments / Interpretacja i nazywanie segmentów

Read `README.md` (or `README.pl.md`) first — it has Aurora Stream's framing for today. Among the solutions Lesson 5 compared, k=2 came out as a strong candidate — the best silhouette score, and robust to swapping out the redundant engagement features. Let's actually look at what those two segments are, and judge the solution partly by whether that profile is sensible and useful.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from task import load_scaled_features

df = load_scaled_features()
df.shape

## What does each cluster actually look like? / Jak właściwie wygląda każdy klaster?

`segment_profiles` fits KMeans at k=2 (using the standardized features internally, so distance is measured fairly) and reports each cluster's average feature values in their original, real-world units, plus how many subscribers landed in it.

In [ ]:
from task import segment_profiles

profiles = segment_profiles(df)
profiles

## Reading the profile table / Odczytywanie tabeli profili

One cluster is much smaller than the other. Compare the three viewing-intensity columns (session_count, total_minutes_watched, avg_minutes_per_session) between the two clusters, then compare tenure_days.

In [ ]:
profiles[["session_count", "total_minutes_watched", "avg_minutes_per_session", "tenure_days"]]

## Does plan tier or country explain the split? / Czy poziom planu lub kraj wyjaśniają podział?

The two clusters separate on viewing engagement — but do they also line up with plan tier or country, features the clustering never saw? If they did, the story would really be "premium vs. basic" or "one region vs. another" in disguise. Check the actual crosstab, don't just assume.

In [ ]:
import sqlite3

import pandas as pd
from sklearn.cluster import KMeans
from task import DATA_PATH, RANDOM_STATE, SCALED_FEATURE_COLUMNS, K

model = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10)
clusters = model.fit_predict(df[SCALED_FEATURE_COLUMNS])
cluster_by_subscriber = df[["subscriber_id"]].assign(cluster=clusters)

conn = sqlite3.connect(DATA_PATH)
tiers = pd.read_sql_query("SELECT subscriber_id, plan_tier, country FROM subscribers", conn)
conn.close()
tiers = tiers.merge(cluster_by_subscriber, on="subscriber_id")

pd.crosstab(tiers["cluster"], tiers["plan_tier"], normalize="index")

In [ ]:
pd.crosstab(tiers["cluster"], tiers["country"], normalize="index")

## Your notes / Twoje notatki

One segment is small and clearly high-engagement; the other is large and clearly low-engagement, and tenure barely differs between them. One sentence: what would you name these two segments, and what would you tell Aurora Stream to do differently for each one?